In [2]:
!pip -q install catboost optuna

In [3]:
import numpy as np
import pandas as pd
import json, time
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import RepeatedKFold, KFold

pd.set_option("display.width", 200)

DATA_PATH = "COCOMO-81.csv"
N_SPLITS, N_REPEATS, SEED = 5, 10, 42
CAT_COLS = ["dev_mode"]
DRIVERS = ["rely", "data", "cplx", "time", "stor", "virt", "turn",
           "acap", "aexp", "pcap", "vexp", "lexp", "modp", "tool", "sced"]

SEERA_FROZEN = {
    "depth": 3,
    "learning_rate": 0.04124173274946173,
    "iterations": 600,
    "l2_leaf_reg": 13.500717070418117,
    "random_strength": 0.41882193959396974,
    "bagging_temperature": 1.9187145799605694,
}
print("Settings ready")

Settings ready


In [4]:
raw = pd.read_csv(DATA_PATH, keep_default_na=False, na_values="?")
df = raw.drop(columns=["num"]).drop_duplicates().reset_index(drop=True)

df["dev_mode"] = df["dev_mode"].fillna("Missing").astype(str).str.replace(r"\.0$", "", regex=True)
for c in df.columns:
    if c != "dev_mode":
        df[c] = pd.to_numeric(df[c], errors="coerce")

y = df["actual"].to_numpy(float)
print("Shape:", df.shape)            # expected (63, 18)
print("Missing cells:", int(df.isna().sum().sum()))
df.head()

Shape: (63, 18)
Missing cells: 0


,dev_mode,rely,data,cplx,time,stor,virt,turn,acap,aexp,pcap,vexp,lexp,modp,tool,sced,loc,actual
0,embedded,0.88,1.16,0.70,1.0,1.06,1.15,1.07,1.19,1.13,1.17,1.1,1.00,1.24,1.10,1.04,113.0,2040.0
1,embedded,0.88,1.16,0.85,1.0,1.06,1.00,1.07,1.00,0.91,1.00,0.9,0.95,1.10,1.00,1.00,293.0,1600.0
2,semidetached,1.00,1.16,0.85,1.0,1.00,0.87,0.94,0.86,0.82,0.86,0.9,0.95,0.91,0.91,1.00,132.0,243.0
3,organic,0.75,1.16,0.70,1.0,1.00,0.87,1.00,1.19,0.91,1.42,1.0,0.95,1.24,1.00,1.04,60.0,240.0
4,organic,0.88,0.94,1.00,1.0,1.00,0.87,1.00,1.00,1.00,0.86,0.9,0.95,1.24,1.00,1.00,16.0,33.0


In [5]:
def random_guess_indices(n, rng):
    idx = rng.integers(0, n, size=n)
    same = idx == np.arange(n)
    while same.any():
        idx[same] = rng.integers(0, n, size=same.sum())
        same = idx == np.arange(n)
    return idx

def random_guess_mae(y_true, seed=SEED, n_boot=1000):
    rng = np.random.default_rng(seed)
    maes = [np.abs(y_true - y_true[random_guess_indices(len(y_true), rng)]).mean() for _ in range(n_boot)]
    return float(np.mean(maes))

RAND_MAE = random_guess_mae(y)

def metrics(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    ae = np.abs(y_true - y_pred)
    mre = ae / y_true
    mae = ae.mean()
    r2 = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)
    return {"MAE": mae, "R2": r2, "MMRE": mre.mean() * 100, "MdMRE": np.median(mre) * 100,
            "PRED25": np.mean(mre <= 0.25) * 100, "SA": (1 - mae / RAND_MAE) * 100,
            "MRE_over100": int((mre > 1).sum())}

print("Random-guess MAE:", round(RAND_MAE, 2))

Random-guess MAE: 1119.92


In [6]:
SPLITS = list(RepeatedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED).split(df))
ALL = {}   # experiment name -> per-repeat metrics

def make_model(params):
    return CatBoostRegressor(**params, loss_function="RMSE", random_seed=SEED, bootstrap_type="Bayesian",
                             thread_count=-1, verbose=0, allow_writing_files=False)

def add_features(d):
    """COCOMO-specific features: log(KLOC), the product of the 15 drivers (EAF), and log(EAF)."""
    d = d.copy()
    d["log_loc"] = np.log(d["loc"])
    d["eaf"] = d[DRIVERS].prod(axis=1)
    d["log_eaf"] = np.log(d["eaf"])
    return d

def run_cv(name, params=SEERA_FROZEN, features="base", threshold=None, weight=1.0):
    X = df.drop(columns=["actual"])
    if features == "fe":
        X = add_features(X)
    ylog = np.log1p(y)
    oof = np.full((N_REPEATS, len(y)), np.nan)
    t0 = time.time()
    for k, (tr, te) in enumerate(SPLITS):
        w = None if threshold is None else np.where(y[tr] < threshold, weight, 1.0)   # TRAIN targets only
        m = make_model(params)
        m.fit(Pool(X.iloc[tr], ylog[tr], cat_features=CAT_COLS, weight=w))
        oof[k // N_SPLITS, te] = np.clip(np.expm1(m.predict(Pool(X.iloc[te], cat_features=CAT_COLS))), 0, None)
    res = pd.DataFrame([metrics(y, oof[r]) for r in range(N_REPEATS)])
    ALL[name] = res
    mean = res.mean()
    print(f"{name:<28} MMRE={mean.MMRE:6.2f}%  MdMRE={mean.MdMRE:6.2f}%  PRED25={mean.PRED25:5.2f}%  "
          f"SA={mean.SA:6.2f}%  R2={mean.R2:5.3f}   ({time.time()-t0:.0f}s)")
    return res

print("Folds:", len(SPLITS))

Folds: 50


In [7]:
run_cv("baseline")

baseline                     MMRE= 58.06%  MdMRE= 37.50%  PRED25=32.54%  SA= 61.27%  R2=0.350   (11s)


,MAE,R2,MMRE,MdMRE,PRED25,SA,MRE_over100
0,464.315050,0.272790,54.431261,40.243582,31.746032,58.540408,8
1,422.256872,0.456920,63.935484,40.405372,31.746032,62.295864,11
2,468.813217,0.240586,57.897556,37.333971,36.507937,58.138758,10
3,416.986345,0.412468,58.651437,38.795671,34.920635,62.766480,10
4,470.101816,0.178460,57.160740,35.084565,31.746032,58.023696,8
5,435.814990,0.344449,59.136633,36.670208,28.571429,61.085233,9
6,406.442773,0.428463,55.828084,35.279410,36.507937,63.707936,8
7,424.620056,0.353638,58.746690,35.888024,33.333333,62.084851,9
8,411.581411,0.407231,57.349207,37.172577,30.158730,63.249097,7
9,416.339549,0.409048,57.511035,38.097402,30.158730,62.824233,9


In [8]:
for thr_name, thr in [("33PM", 33), ("100PM", 100)]:
    for w in [2, 3, 4]:
        run_cv(f"weight_{w}x_below_{thr_name}", threshold=thr, weight=float(w))

weight_2x_below_33PM         MMRE= 57.33%  MdMRE= 39.06%  PRED25=31.75%  SA= 60.16%  R2=0.317   (11s)
weight_3x_below_33PM         MMRE= 57.72%  MdMRE= 39.33%  PRED25=31.75%  SA= 59.67%  R2=0.303   (11s)
weight_4x_below_33PM         MMRE= 57.43%  MdMRE= 41.24%  PRED25=31.27%  SA= 58.90%  R2=0.271   (10s)
weight_2x_below_100PM        MMRE= 57.71%  MdMRE= 39.12%  PRED25=31.90%  SA= 59.59%  R2=0.302   (11s)
weight_3x_below_100PM        MMRE= 57.26%  MdMRE= 40.48%  PRED25=30.16%  SA= 58.39%  R2=0.261   (11s)
weight_4x_below_100PM        MMRE= 58.12%  MdMRE= 42.30%  PRED25=30.32%  SA= 57.27%  R2=0.227   (11s)


In [9]:
run_cv("cocomo_features", features="fe")

cocomo_features              MMRE= 49.86%  MdMRE= 37.25%  PRED25=34.60%  SA= 64.18%  R2=0.408   (12s)


,MAE,R2,MMRE,MdMRE,PRED25,SA,MRE_over100
0,464.166924,0.240187,48.673793,40.696014,33.333333,58.553634,6
1,371.868502,0.504180,51.499930,35.915089,34.920635,66.795139,8
2,453.773705,0.212339,48.009254,37.145961,30.158730,59.481665,4
3,380.892818,0.497577,48.534459,33.652297,36.507937,65.989341,6
4,461.594148,0.189561,53.579000,37.848221,33.333333,58.783362,5
5,399.942748,0.428143,51.768356,40.239928,34.920635,64.288335,5
6,356.043087,0.546201,48.124411,34.350961,39.682540,68.208222,8
7,376.069186,0.463433,49.732233,37.498923,36.507937,66.420052,7
8,382.099634,0.467803,49.159676,37.861831,31.746032,65.881582,6
9,365.667475,0.527511,49.490299,37.285858,34.920635,67.348841,6


In [10]:
w_names = [n for n in ALL if n.startswith("weight_")]
best_w = max(w_names, key=lambda n: ALL[n]["SA"].mean())
print("Best weighting so far (by SA):", best_w)

weight_val = float(best_w.split("_")[1].replace("x", ""))
thr_val = 33 if best_w.endswith("33PM") else 100
run_cv("features_plus_" + best_w, features="fe", threshold=thr_val, weight=weight_val)

Best weighting so far (by SA): weight_2x_below_33PM
features_plus_weight_2x_below_33PM MMRE= 50.05%  MdMRE= 37.92%  PRED25=33.17%  SA= 62.48%  R2=0.356   (12s)


,MAE,R2,MMRE,MdMRE,PRED25,SA,MRE_over100
0,472.529629,0.204764,49.326332,34.823792,36.507937,57.806912,6
1,402.729999,0.427235,50.996011,35.270436,38.095238,64.039456,6
2,470.951386,0.177838,49.090637,36.624245,22.222222,57.947836,4
3,400.481493,0.433509,49.148503,36.706370,33.333333,64.240230,6
4,467.782150,0.172109,52.200054,43.424584,26.984127,58.230824,5
5,422.457614,0.345493,51.135353,42.816388,33.333333,62.277939,5
6,384.312805,0.457472,47.477765,35.973396,41.269841,65.683963,5
7,396.896858,0.422045,51.690600,36.409902,33.333333,64.560309,6
8,394.266858,0.441776,49.104947,38.480175,30.158730,64.795147,4
9,389.591751,0.481602,50.320369,38.696504,36.507937,65.212596,5


In [11]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def tune(X_tr, y_tr, n_trials=12, show=True):
    ylog_tr = np.log1p(y_tr)
    folds = list(KFold(n_splits=5, shuffle=True, random_state=123).split(X_tr))

    def objective(trial):
        p = {
            "depth": trial.suggest_int("depth", 3, 6),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
            "iterations": trial.suggest_int("iterations", 300, 1000, step=100),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 5, 20, log=True),
            "random_strength": trial.suggest_float("random_strength", 0.1, 3.0, log=True),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.5, 2),
        }
        pred = np.zeros(len(y_tr))
        for a, b in folds:
            m = make_model(p)
            m.fit(Pool(X_tr.iloc[a], ylog_tr[a], cat_features=CAT_COLS))
            pred[b] = np.clip(np.expm1(m.predict(Pool(X_tr.iloc[b], cat_features=CAT_COLS))), 0, None)
        return float((np.abs(y_tr - pred) / y_tr).mean())

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=show)
    return dict(study.best_trial.params), study

X_base = df.drop(columns=["actual"])
best_params, study = tune(X_base, y)
print("Best parameters:")
print(json.dumps(best_params, indent=2))
print("Best tuning MMRE: %.2f%%" % (study.best_value * 100))

run_cv("tuned_optuna", params=best_params)
run_cv("tuned_optuna_plus_features", params=best_params, features="fe")

  0%|          | 0/12 [00:00<?, ?it/s]

Best parameters:
{
  "depth": 3,
  "learning_rate": 0.06642255121457301,
  "iterations": 900,
  "l2_leaf_reg": 9.932918364573233,
  "random_strength": 0.10176198224862719,
  "bagging_temperature": 1.6947694190032105
}
Best tuning MMRE: 59.92%
tuned_optuna                 MMRE= 56.32%  MdMRE= 38.73%  PRED25=31.59%  SA= 62.90%  R2=0.400   (15s)
tuned_optuna_plus_features   MMRE= 48.11%  MdMRE= 35.31%  PRED25=36.19%  SA= 65.47%  R2=0.437   (23s)


,MAE,R2,MMRE,MdMRE,PRED25,SA,MRE_over100
0,468.134833,0.236605,49.707986,41.776544,31.746032,58.199332,8
1,340.353892,0.587563,51.943513,34.263229,38.095238,69.609140,10
2,433.223520,0.271250,47.080433,35.243214,33.333333,61.316631,5
3,369.002816,0.528121,46.611554,33.177755,36.507937,67.051022,5
4,451.679721,0.218819,50.955899,32.188721,42.857143,59.668641,6
5,387.105956,0.437474,48.393184,34.624083,33.333333,65.434558,4
6,346.305820,0.549509,44.310354,32.384571,42.857143,69.077681,4
7,388.310191,0.465021,49.450752,41.067493,34.920635,65.327029,6
8,333.471476,0.541427,46.294786,33.229815,33.333333,70.223684,6
9,349.539957,0.536728,46.343581,35.096317,34.920635,68.788899,6


In [12]:
RUN_NESTED = True
NESTED_REPEATS = 2

if RUN_NESTED:
    X = df.drop(columns=["actual"])
    ylog = np.log1p(y)
    n_use = N_SPLITS * NESTED_REPEATS
    oof_n = np.full((NESTED_REPEATS, len(y)), np.nan)
    oof_f = np.full((NESTED_REPEATS, len(y)), np.nan)

    for k, (tr, te) in enumerate(SPLITS[:n_use]):
        print(f"Nested fold {k + 1}/{n_use}", flush=True)
        best, _ = tune(X.iloc[tr].reset_index(drop=True), y[tr], show=False)
        for params, store in [(best, oof_n), (SEERA_FROZEN, oof_f)]:
            m = make_model(params)
            m.fit(Pool(X.iloc[tr], ylog[tr], cat_features=CAT_COLS))
            store[k // N_SPLITS, te] = np.clip(np.expm1(m.predict(Pool(X.iloc[te], cat_features=CAT_COLS))), 0, None)

    nested = pd.DataFrame([metrics(y, oof_n[r]) for r in range(NESTED_REPEATS)]).mean()
    frozen = pd.DataFrame([metrics(y, oof_f[r]) for r in range(NESTED_REPEATS)]).mean()
    out = pd.DataFrame({"SEERA_frozen (same folds)": frozen, "nested_tuned": nested}).round(3)
    print("\nNESTED (honest) result, mean over", NESTED_REPEATS, "repeats:")
    print(out.to_string())

Nested fold 1/10
Nested fold 2/10
Nested fold 3/10
Nested fold 4/10
Nested fold 5/10
Nested fold 6/10
Nested fold 7/10
Nested fold 8/10
Nested fold 9/10
Nested fold 10/10

NESTED (honest) result, mean over 2 repeats:
             SEERA_frozen (same folds)  nested_tuned
MAE                            443.286       442.476
R2                               0.365         0.367
MMRE                            59.183        58.637
MdMRE                           40.324        39.182
PRED25                          31.746        36.508
SA                              60.418        60.490
MRE_over100                      9.500        10.000


In [13]:
cols = ["MMRE", "MdMRE", "PRED25", "SA", "R2", "MAE", "MRE_over100"]
summary = pd.DataFrame({n: r[cols].mean() for n, r in ALL.items()}).T
base = ALL["baseline"]
summary["SA_better_in"] = [int((ALL[n]["SA"] > base["SA"]).sum()) for n in summary.index]
summary["MMRE_better_in"] = [int((ALL[n]["MMRE"] < base["MMRE"]).sum()) for n in summary.index]
summary = summary.sort_values("SA", ascending=False).round(2)
summary.to_csv("experiments_summary.csv")
pd.concat([r.assign(experiment=n) for n, r in ALL.items()]).to_csv("experiments_per_repeat.csv", index=False)
summary

,MMRE,MdMRE,PRED25,SA,R2,MAE,MRE_over100,SA_better_in,MMRE_better_in
tuned_optuna_plus_features,48.11,35.31,36.19,65.47,0.44,386.71,6.0,9,10
cocomo_features,49.86,37.25,34.60,64.18,0.41,401.21,6.1,10,10
tuned_optuna,56.32,38.73,31.59,62.90,0.40,415.47,8.8,9,8
features_plus_weight_2x_below_33PM,50.05,37.92,33.17,62.48,0.36,420.20,5.2,8,10
baseline,58.06,37.50,32.54,61.27,0.35,433.73,8.9,0,0
weight_2x_below_33PM,57.33,39.06,31.75,60.16,0.32,446.16,8.5,0,6
weight_3x_below_33PM,57.72,39.33,31.75,59.67,0.30,451.70,8.0,0,5
weight_2x_below_100PM,57.71,39.12,31.90,59.59,0.30,452.53,8.3,0,6
weight_4x_below_33PM,57.43,41.24,31.27,58.90,0.27,460.28,7.3,0,5
weight_3x_below_100PM,57.26,40.48,30.16,58.39,0.26,466.04,7.8,0,5


In [14]:
import zipfile
with zipfile.ZipFile("cocomo_experiments_results.zip", "w") as z:
    for f in ["experiments_summary.csv", "experiments_per_repeat.csv"]:
        z.write(f)
try:
    from google.colab import files
    files.download("cocomo_experiments_results.zip")
except ImportError:
    print("Saved cocomo_experiments_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>